### Data cleaning notebook

In [0]:
BRONZE_TABLE = "workspace.new_project.bronze_online_retail"
SILVER_TABLE = "workspace.new_project.silver_online_retail"

In [0]:
from pyspark.sql.functions import (
    col, to_timestamp, upper, trim, current_timestamp
)

In [0]:
df_bronze = spark.read.table(BRONZE_TABLE)
initial_count = df_bronze.count()
print(f"Bronze rows: {initial_count:,}")

Bronze rows: 1,067,371


# Rename columns

In [0]:
df = df_bronze \
    .withColumnRenamed("Invoice",     "invoice_id") \
    .withColumnRenamed("StockCode",   "stock_code") \
    .withColumnRenamed("Description", "description") \
    .withColumnRenamed("Quantity",    "quantity") \
    .withColumnRenamed("InvoiceDate", "invoice_date") \
    .withColumnRenamed("Price",       "unit_price") \
    .withColumnRenamed("Customer_ID", "customer_id") \
    .withColumnRenamed("Country",     "country")

# Cast types

In [0]:
df = df \
    .withColumn("invoice_date", to_timestamp(col("invoice_date"), "M/d/yyyy H:mm")) \
    .withColumn("quantity",    col("quantity").cast("integer")) \
    .withColumn("unit_price",  col("unit_price").cast("double")) \
    .withColumn("customer_id", col("customer_id").cast("long"))

# Remove nulls in critical columns

In [0]:
df = df.dropna(subset=["invoice_id", "customer_id", "stock_code"])

# Remove duplicates

In [0]:
df = df.dropDuplicates(["invoice_id", "stock_code"])

# Business rule filters

In [0]:
df = df.filter(
    (col("quantity")   > 0) &
    (col("unit_price") > 0) &
    (~col("invoice_id").startswith("C"))
)

# Add derived columns

In [0]:
df_silver = df \
    .withColumn("total_amount", col("quantity") * col("unit_price")) \
    .withColumn("country", upper(trim(col("country"))))
final_count = df_silver.count()
print(f"Silver rows: {final_count:,} (removed {initial_count - final_count:,} rows)")

Silver rows: 768,862 (removed 298,509 rows)


# Write as Unity Catalog table (partitioning optional — keep it simple first)

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_TABLE)

print(f"✅ Silver table written → {SILVER_TABLE}")

✅ Silver table written → workspace.new_project.silver_online_retail
